# Lib


In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Data


In [3]:
df = pd.read_csv("../data/clean/data.csv", parse_dates=["date"])
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df = df.sort_values(["date", "location_id"]).reset_index(drop=True)
print("Shape:", df.shape)
print("Time range:", df["date"].min(), "->", df["date"].max())
df.head()

Shape: (306000, 30)
Time range: 2023-01-01 00:00:00 -> 2026-06-28 23:00:00


,location_id,location_name,date,pm10,pm2_5,carbon_monoxide,sulphur_dioxide,ozone,nitrogen_dioxide,aerosol_optical_depth,...,rain,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,weather_code,sunshine_duration,boundary_layer_height,dew_point_2m,is_day
0,1562414,Vũng Tàu,2023-01-01,35.6,22.2,483.0,5.40,99.0,13.55,0.29,...,0.0,1012.49176,100.0,10.661107,101.689350,3.0,0.0,135.0,22.15,0.0
1,1565022,Thu Dau Mot,2023-01-01,95.0,65.0,993.0,37.50,25.0,84.40,0.29,...,0.0,1012.89570,100.0,11.275530,16.699326,3.0,0.0,250.0,15.55,0.0
2,1566083,Ho Chi Minh City,2023-01-01,95.0,65.0,993.0,37.50,25.0,84.40,0.29,...,0.0,1013.40027,100.0,11.304229,9.162280,3.0,0.0,70.0,16.00,0.0
3,1566559,Tay Ninh,2023-01-01,22.9,14.9,409.0,0.95,80.0,1.95,0.19,...,0.0,1012.49800,100.0,12.661564,14.826528,3.0,0.0,620.0,15.55,0.0
4,1582436,Dong Xoai,2023-01-01,16.9,10.5,393.0,0.90,95.0,1.10,0.12,...,0.0,1003.73650,100.0,11.503113,20.136398,3.0,0.0,475.0,15.10,0.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 306000 entries, 0 to 305999
Data columns (total 30 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   location_id              306000 non-null  int64         
 1   location_name            306000 non-null  str           
 2   date                     306000 non-null  datetime64[us]
 3   pm10                     306000 non-null  float64       
 4   pm2_5                    306000 non-null  float64       
 5   carbon_monoxide          306000 non-null  float64       
 6   sulphur_dioxide          306000 non-null  float64       
 7   ozone                    306000 non-null  float64       
 8   nitrogen_dioxide         306000 non-null  float64       
 9   aerosol_optical_depth    306000 non-null  float64       
 10  dust                     306000 non-null  float64       
 11  us_aqi_pm2_5             306000 non-null  float64       
 12  us_aqi_pm10              30

# Feature Engineering


## Label sample


In [5]:
def aqi_to_category(value):
    if value <= 50:
        return "good"
    elif value <= 100:
        return "moderate"
    elif value <= 150:
        return "unhealthy for sensitive groups"
    elif value <= 200:
        return "unhealthy"
    elif value <= 300:
        return "very unhealthful"
    else:
        return "hazardous"


category_order = [
    "good",
    "moderate",
    "unhealthy for sensitive groups",
    "unhealthy",
    "very unhealthful",
    "hazardous",
]

df["aqi_category"] = df["us_aqi"].apply(aqi_to_category)

category_counts = df["aqi_category"].value_counts()
category_counts.reset_index()

,aqi_category,count
0,moderate,213466
1,good,54488
2,unhealthy for sensitive groups,33585
3,unhealthy,4303
4,very unhealthful,158


In [6]:
MIN_SAMPLES_PER_CLASS = 30  # Threshold for minimum samples per class

rare_classes = category_counts[category_counts < MIN_SAMPLES_PER_CLASS].index.tolist()
if rare_classes:
    print(
        f"Classes with fewer than {MIN_SAMPLES_PER_CLASS} samples will be grouped into 'Very Unhealthy': {rare_classes}"
    )
    df["aqi_category"] = df["aqi_category"].replace(
        {c: "Very Unhealthy" for c in rare_classes if c != "Very Unhealthy"}
    )

df["aqi_category"].value_counts().reset_index()

,aqi_category,count
0,moderate,213466
1,good,54488
2,unhealthy for sensitive groups,33585
3,unhealthy,4303
4,very unhealthful,158


## Impute missing for `boundary_layer_height`


In [7]:
missing_before = df["boundary_layer_height"].isnull().sum()
print(
    f"Missing boundary_layer_height before impute: {missing_before} ({missing_before/len(df)*100:.2f}%)"
)

df = df.sort_values(["location_id", "date"])
df["boundary_layer_height"] = (
    df.groupby("location_id")["boundary_layer_height"]
    .apply(lambda s: s.interpolate(method="linear", limit_direction="both"))
    .reset_index(drop=True)
)

missing_after = df["boundary_layer_height"].isnull().sum()
print(
    f"Missing boundary_layer_height sau impute   : {missing_after} ({missing_after/len(df)*100:.2f}%)"
)

Missing boundary_layer_height before impute: 43680 (14.27%)
Missing boundary_layer_height sau impute   : 0 (0.00%)


## Time features


In [8]:
df["hour"] = df["date"].dt.hour
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df["season"] = np.where(
    df["month"].isin([11, 12, 1, 2, 3, 4]), 0, 1
)  # 0 = dry season, 1 = wet season

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df[
    [
        "date",
        "hour",
        "month",
        "day_of_week",
        "is_weekend",
        "season",
        "hour_sin",
        "hour_cos",
    ]
].head()

,date,hour,month,day_of_week,is_weekend,season,hour_sin,hour_cos
0,2023-01-01 00:00:00,0,1,6,1,0,0.000000,1.000000
10,2023-01-01 01:00:00,1,1,6,1,0,0.258819,0.965926
20,2023-01-01 02:00:00,2,1,6,1,0,0.500000,0.866025
30,2023-01-01 03:00:00,3,1,6,1,0,0.707107,0.707107
40,2023-01-01 04:00:00,4,1,6,1,0,0.866025,0.500000


## Lag Feature


In [9]:
df = df.sort_values(["location_id", "date"])
df["us_aqi_lag1"] = df.groupby("location_id")["us_aqi"].shift(1)
df["us_aqi_lag3"] = df.groupby("location_id")["us_aqi"].shift(3)

# First row of each location will be NaN in lag feature (no previous hour) -> acceptable,
# Will drop these rows before training (very small number: 10 locations x 3 lag = maximum 30 rows).
print(
    "Rows with NaN in lag features:",
    df["us_aqi_lag1"].isnull().sum() + df["us_aqi_lag3"].isnull().sum(),
)

Rows with NaN in lag features: 40


In [10]:
df = df.dropna(subset=["us_aqi_lag1", "us_aqi_lag3"]).reset_index(drop=True)
print("Shape after dropping rows with NaN in lag features:", df.shape)

Shape after dropping rows with NaN in lag features: (305970, 40)


# Train, test split


In [11]:
TRAIN_RATIO = 0.8

In [12]:
df = df.sort_values("date").reset_index(drop=True)

unique_timestamps = sorted(df["date"].unique())
cutoff_idx = int(len(unique_timestamps) * TRAIN_RATIO)
cutoff_date = unique_timestamps[cutoff_idx]

print(f"Total unique timestamps: {len(unique_timestamps)}")
print(f"Train/Test ratio       : {TRAIN_RATIO:.0%} / {1 - TRAIN_RATIO:.0%}")
print(f"Cutoff date            : {cutoff_date}")

Total unique timestamps: 30597
Train/Test ratio       : 80% / 20%
Cutoff date            : 2025-10-17 00:00:00


In [13]:
train_df = df[df["date"] < cutoff_date].reset_index(drop=True)
test_df = df[df["date"] >= cutoff_date].reset_index(drop=True)

print(
    f"Train: {train_df.shape[0]:<8,} rows | {train_df['date'].min()} -> {train_df['date'].max()}"
)
print(
    f"Test : {test_df.shape[0]:<8,} rows | {test_df['date'].min()} -> {test_df['date'].max()}"
)
print(
    f"Actual ratio: Train {len(train_df)/len(df)*100:.1f}% / Test {len(test_df)/len(df)*100:.1f}%"
)

Train: 244,770  rows | 2023-01-01 03:00:00 -> 2025-10-16 23:00:00
Test : 61,200   rows | 2025-10-17 00:00:00 -> 2026-06-28 23:00:00
Actual ratio: Train 80.0% / Test 20.0%


In [14]:
# Check for time overlap between train and test sets
assert (
    train_df["date"].max() < test_df["date"].min()
), "ERROR: There is a time overlap between Train and Test!"
print("Confirmation successful: No time overlap between train set and test set.")

Confirmation successful: No time overlap between train set and test set.


In [15]:
print("Distribution of labels - train:")
print(train_df["aqi_category"].value_counts(normalize=True).round(4))

Distribution of labels - train:
aqi_category
moderate                          0.6988
good                              0.1920
unhealthy for sensitive groups    0.0976
unhealthy                         0.0112
very unhealthful                  0.0003
Name: proportion, dtype: float64


In [16]:
print("Distribution of labels - test:")
print(test_df["aqi_category"].value_counts(normalize=True).round(4))

Distribution of labels - test:
aqi_category
moderate                          0.6931
unhealthy for sensitive groups    0.1581
good                              0.1223
unhealthy                         0.0253
very unhealthful                  0.0013
Name: proportion, dtype: float64


# Save


In [17]:
feature_cols = [
    "location_id",
    "location_name",
    "date",
    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "weather_code",
    "sunshine_duration",
    "boundary_layer_height",
    "dew_point_2m",
    "hour",
    "month",
    "day_of_week",
    "is_weekend",
    "season",
    "hour_sin",
    "hour_cos",
    "us_aqi_lag1",
    "us_aqi_lag3",
    "us_aqi",
    "aqi_category",
]

In [18]:
train_out = train_df[feature_cols]
test_out = test_df[feature_cols]

train_path = "../data/model/train.csv"
test_path = "../data/model/test.csv"

train_out.to_csv(train_path, index=False)
test_out.to_csv(test_path, index=False)